# Correccion de Doble Normalizacion — Dataset Extendido

**Problema:** `master_dataset_extendido.csv` mezcla valores crudos (2019-2020) con z-scores de fase 2 (2021-2025). El `master_escalado_extendido.csv` aplico StandardScaler sobre esa mezcla, produciendo una doble normalizacion para 2021-2025 que comprime artificialmente la varianza y genera MAEs irrealmente bajos.

**Solucion:**
1. Desnormalizar las columnas 2021-2025 usando el scaler original de fase 2 (`notebooks/fase2/scalers/standard_scaler_fase2.joblib`)
2. Con todo en escala cruda, agregar a nivel mensual, incorporar NLP
3. Aplicar UNA SOLA normalizacion (StandardScaler fit en train)
4. Guardar `master_escalado_extendido_v2.csv`

In [1]:
import numpy as np
import pandas as pd
import json, joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'CLAUDE.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

scaler_fase2 = joblib.load(ROOT / 'notebooks/fase2/scalers/standard_scaler_fase2.joblib')
cols_fase2   = list(joblib.load(ROOT / 'notebooks/fase2/scalers/cols_to_scale.joblib'))
NLP_PATH     = ROOT / 'notebooks/fase2/output/01_nlp_sentimiento/sentimiento_mensual_v2.csv'

print(f'Raiz: {ROOT}')
print(f'Scaler fase2: {len(cols_fase2)} columnas')
print(f'NLP: {NLP_PATH.exists()}')

Raiz: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-
Scaler fase2: 51 columnas
NLP: True


## PASO 1 — Desnormalizar 2021-2025 a escala cruda

Usar el scaler original de fase 2 para invertir los z-scores de las columnas que fueron escaladas.
Las columnas no escaladas (lat, lon, ciclicas) ya estan en escala natural y no se tocan.

In [2]:
# PASO 1: Cargar y desnormalizar
df = pd.read_csv(ROOT / 'data/processed/master_dataset_extendido.csv')
print(f'master_dataset_extendido.csv: {df.shape}')

mask_2125 = df['fecha_evento'].str[:4].isin(['2021','2022','2023','2024','2025'])
mask_1920 = ~mask_2125

# Columnas del extended dataset que tienen z-scores en 2021-2025
# (cruzar con el scaler de fase2)
COLS_EXTENDED = df.columns.tolist()
COLS_ZSCORED = [c for c in COLS_EXTENDED if c in cols_fase2]
print(f'\nColumnas z-scored en fase2 presentes en extended ({len(COLS_ZSCORED)}): {COLS_ZSCORED}')

# hectareas_cultivo_perdidas esta z-scored pero NO en el scaler de fase2
# Estimar sus parametros desde los datos crudos disponibles
# El archivo INDECI 2021-2025 tiene los valores raw a nivel provincia-mes
hect_col = 'hectareas_cultivo_perdidas'
print(f'\n{hect_col}:')
print(f'  2019-20 (crudo): mean={df.loc[mask_1920, hect_col].mean():.2f}  std={df.loc[mask_1920, hect_col].std():.2f}')
print(f'  2021-25 (z-score): mean={df.loc[mask_2125, hect_col].mean():.4f}  std={df.loc[mask_2125, hect_col].std():.4f}')

# Invertir z-scores de las 12 columnas con el scaler de fase2
scaler_means = dict(zip(cols_fase2, scaler_fase2.mean_))
scaler_scales = dict(zip(cols_fase2, scaler_fase2.scale_))

for col in COLS_ZSCORED:
    mean_orig = scaler_means[col]
    scale_orig = scaler_scales[col]
    df.loc[mask_2125, col] = df.loc[mask_2125, col] * scale_orig + mean_orig
    print(f'  {col:35s} z*{scale_orig:.4f} + {mean_orig:.4f}')

# hectareas: estimar scaler desde datos crudos del pipeline INDECI
# Reconstruir a nivel provincia-mes para el grid completo (105 provs * 56 meses = 5880)
indeci_raw = pd.read_csv(ROOT / 'data/interim/indeci/indeci_temporal_2021_2025.csv')
hect_vals = indeci_raw[hect_col].values
n_grid = mask_2125.sum()
n_indeci = len(indeci_raw)
hect_all = np.concatenate([hect_vals, np.zeros(max(0, n_grid - n_indeci))])
hect_mean = float(hect_all.mean())
hect_std  = float(hect_all.std())
df.loc[mask_2125, hect_col] = df.loc[mask_2125, hect_col] * hect_std + hect_mean
print(f'  {hect_col:35s} z*{hect_std:.4f} + {hect_mean:.4f}  (estimado)')

# Verificar que ahora ambos periodos estan en escala cruda comparable
print(f'\n--- Verificacion post-desnormalizacion ---')
for col in ['produccion_t', 'precio_chacra_kg', 'T2M', 'PRECTOTCORR', hect_col]:
    v19 = df.loc[mask_1920, col]
    v21 = df.loc[mask_2125, col]
    print(f'  {col:35s}  19-20: mean={v19.mean():10.2f} std={v19.std():10.2f}  |  21-25: mean={v21.mean():10.2f} std={v21.std():10.2f}')

master_dataset_extendido.csv: (7707, 24)

Columnas z-scored en fase2 presentes en extended (12): ['produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M']

hectareas_cultivo_perdidas:
  2019-20 (crudo): mean=12.57  std=122.68
  2021-25 (z-score): mean=0.0000  std=1.0001
  produccion_t                        z*27.9869 + 16.3198
  precio_chacra_kg                    z*0.5962 + 1.5584
  num_emergencias                     z*3.0912 + 0.8314
  total_afectados                     z*181.2488 + 15.4578
  ALLSKY_SFC_SW_DWN                   z*2.9644 + 17.9182
  PRECTOTCORR                         z*72.9954 + 51.5703
  QV2M                                z*3.5718 + 11.5610
  RH2M                                z*11.8383 + 70.9292
  T2M                                 z*6.3780 + 18.7497
  T2M_MAX                             z*5.7099 + 27.4818
  T2M_MIN                             z*7.5367 + 1

## PASO 2 — Agregar a nivel mensual, incorporar NLP, normalizar UNA sola vez

Con todos los valores en escala cruda:
1. Agregar por `fecha_evento` (media de provincias)
2. Incorporar NLP (nlp_index + lag, tiene_nlp, es_shock)
3. Split: train = primeros 68 meses, test = ultimos 12
4. StandardScaler fit en train, transform ambos

In [3]:
# PASO 2a: Agregar a nivel mensual (todo crudo ahora)
df['fecha_evento'] = pd.to_datetime(df['fecha_evento'])
df_monthly = df.groupby('fecha_evento').mean(numeric_only=True).reset_index()
df_monthly = df_monthly.sort_values('fecha_evento').reset_index(drop=True)

print(f'Agregado mensual: {df_monthly.shape}')
print(f'Rango: {df_monthly["fecha_evento"].min().date()} -> {df_monthly["fecha_evento"].max().date()}')

# Verificar que la escala es comparable entre periodos
mask_m19 = df_monthly['fecha_evento'].dt.year.isin([2019, 2020])
mask_m21 = df_monthly['fecha_evento'].dt.year >= 2021
TARGET = 'produccion_t'

print(f'\n--- produccion_t mensual (crudo) ---')
print(f'  2019-20: mean={df_monthly.loc[mask_m19, TARGET].mean():.2f}  std={df_monthly.loc[mask_m19, TARGET].std():.2f}')
print(f'  2021-25: mean={df_monthly.loc[mask_m21, TARGET].mean():.2f}  std={df_monthly.loc[mask_m21, TARGET].std():.2f}')
print(f'  Global:  mean={df_monthly[TARGET].mean():.2f}  std={df_monthly[TARGET].std():.2f}')

# PASO 2b: Incorporar NLP sentiment
df_nlp = pd.read_csv(NLP_PATH, encoding='utf-8-sig')
fc = [c for c in df_nlp.columns if any(k in c.lower() for k in ['fecha', 'periodo'])][0]
df_nlp = df_nlp.rename(columns={fc: 'fecha_evento'})
df_nlp['fecha_evento'] = pd.to_datetime(df_nlp['fecha_evento'])
df_nlp['nlp_index']      = df_nlp['avg_sentiment'] * np.log1p(df_nlp['n_noticias_beto'])
df_nlp['nlp_index_lag1'] = df_nlp['nlp_index'].shift(1).fillna(0)

df_monthly = df_monthly.merge(df_nlp[['fecha_evento', 'nlp_index', 'nlp_index_lag1']],
                               on='fecha_evento', how='left')
df_monthly['nlp_index']      = df_monthly['nlp_index'].fillna(0)
df_monthly['nlp_index_lag1'] = df_monthly['nlp_index_lag1'].fillna(0)
df_monthly['tiene_nlp'] = (df_monthly['fecha_evento'].dt.year >= 2021).astype(int)

# es_shock sobre valores CRUDOS
df_monthly['es_shock'] = (df_monthly[TARGET].pct_change().abs() > 0.20).astype(int)
df_monthly.loc[0, 'es_shock'] = 0

print(f'\nDataset mensual con NLP: {df_monthly.shape}')
print(f'Columnas: {df_monthly.columns.tolist()}')

Agregado mensual: (80, 22)
Rango: 2019-01-01 -> 2025-08-01

--- produccion_t mensual (crudo) ---
  2019-20: mean=332.16  std=83.39
  2021-25: mean=16.32  std=1.97
  Global:  mean=111.07  std=152.45

Dataset mensual con NLP: (80, 26)
Columnas: ['fecha_evento', 'produccion_t', 'precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos', 'nlp_index', 'nlp_index_lag1', 'tiene_nlp', 'es_shock']


In [4]:
# PASO 2c: Split y normalizacion UNICA
n_total = len(df_monthly)
n_test  = 12
n_train = n_total - n_test

df_train = df_monthly.iloc[:n_train].copy()
df_test  = df_monthly.iloc[n_train:].copy()

print(f'n_train={n_train}  n_test={n_test}')
print(f'Train: {df_train["fecha_evento"].min().date()} -> {df_train["fecha_evento"].max().date()}')
print(f'Test:  {df_test["fecha_evento"].min().date()} -> {df_test["fecha_evento"].max().date()}')

# Columnas a escalar (excluir fecha, flags, target)
NO_SCALE = ['fecha_evento', 'tiene_nlp', 'es_shock', TARGET]
COLS_SCALE = [c for c in df_monthly.columns if c not in NO_SCALE]

print(f'\nColumnas a escalar ({len(COLS_SCALE)}): {COLS_SCALE}')

# StandardScaler fit SOLO en train
scaler_v2 = StandardScaler()
scaler_v2.fit(df_train[COLS_SCALE])

# Scaler separado para target
scaler_y_v2 = StandardScaler()
scaler_y_v2.fit(df_train[[TARGET]])

# Transformar
df_train_sc = df_train.copy()
df_test_sc  = df_test.copy()

df_train_sc[COLS_SCALE] = scaler_v2.transform(df_train[COLS_SCALE])
df_test_sc[COLS_SCALE]  = scaler_v2.transform(df_test[COLS_SCALE])
df_train_sc[TARGET] = scaler_y_v2.transform(df_train[[TARGET]])
df_test_sc[TARGET]  = scaler_y_v2.transform(df_test[[TARGET]])

print(f'\nEscalado completado.')
print(f'Train target: mean={df_train_sc[TARGET].mean():.6f}  std={df_train_sc[TARGET].std():.4f}')
print(f'Test target:  mean={df_test_sc[TARGET].mean():.6f}  std={df_test_sc[TARGET].std():.4f}')

n_train=68  n_test=12
Train: 2019-01-01 -> 2024-08-01
Test:  2024-09-01 -> 2025-08-01

Columnas a escalar (22): ['precio_chacra_kg', 'num_emergencias', 'total_afectados', 'hectareas_cultivo_perdidas', 'ALLSKY_SFC_SW_DWN', 'PRECTOTCORR', 'QV2M', 'RH2M', 'T2M', 'T2M_MAX', 'T2M_MIN', 'WS2M', 'lat', 'lon', 'month_sin', 'month_cos', 'mes_num', 'trimestre_num', 'trimestre_sin', 'trimestre_cos', 'nlp_index', 'nlp_index_lag1']

Escalado completado.
Train target: mean=-0.000000  std=1.0074
Test target:  mean=-0.715306  std=0.0073


## PASO 3 — Verificar integridad

- media~0 y std~1 en train para features escaladas
- Comparar distribucion 2019-2020 vs 2021-2024 dentro del train (deben ser comparables)
- Confirmar que el target tiene rango razonable

In [5]:
# PASO 3: Verificaciones
dataset_v2 = pd.concat([df_train_sc, df_test_sc], ignore_index=True)

print('='*70)
print('VERIFICACIONES DE INTEGRIDAD')
print('='*70)

# 3a. Media y std en train
print(f'\n[1] Train features escaladas (deben tener mean~0, std~1):')
for col in COLS_SCALE[:5]:
    m = df_train_sc[col].mean()
    s = df_train_sc[col].std()
    ok = 'OK' if abs(m) < 0.01 and abs(s - 1) < 0.1 else 'WARN'
    print(f'  {col:30s}  mean={m:+.6f}  std={s:.4f}  {ok}')
print(f'  ... ({len(COLS_SCALE) - 5} columnas mas)')

# 3b. Comparar periodos dentro del train
mask_tr19 = df_train_sc['fecha_evento'].dt.year.isin([2019, 2020])
mask_tr21 = df_train_sc['fecha_evento'].dt.year >= 2021

print(f'\n[2] Distribucion del TARGET (escalado) por periodo en train:')
print(f'  2019-20 ({mask_tr19.sum()} meses): mean={df_train_sc.loc[mask_tr19, TARGET].mean():.4f}  std={df_train_sc.loc[mask_tr19, TARGET].std():.4f}')
print(f'  2021-24 ({mask_tr21.sum()} meses): mean={df_train_sc.loc[mask_tr21, TARGET].mean():.4f}  std={df_train_sc.loc[mask_tr21, TARGET].std():.4f}')

# 3c. Comparar features clave por periodo
print(f'\n[3] Features clave por periodo (escalado):')
for col in ['precio_chacra_kg', 'T2M', 'PRECTOTCORR', 'num_emergencias']:
    v19 = df_train_sc.loc[mask_tr19, col]
    v21 = df_train_sc.loc[mask_tr21, col]
    print(f'  {col:30s}  19-20: mean={v19.mean():+.3f}  21-24: mean={v21.mean():+.3f}')

# 3d. Target desnormalizado (escala real)
print(f'\n[4] Target en escala real (toneladas, media provincial mensual):')
print(f'  Scaler: mean={scaler_y_v2.mean_[0]:.2f}  scale={scaler_y_v2.scale_[0]:.2f}')
target_real_train = df_train[TARGET]
target_real_test  = df_test[TARGET]
print(f'  Train: mean={target_real_train.mean():.2f}  std={target_real_train.std():.2f}  min={target_real_train.min():.2f}  max={target_real_train.max():.2f}')
print(f'  Test:  mean={target_real_test.mean():.2f}  std={target_real_test.std():.2f}  min={target_real_test.min():.2f}  max={target_real_test.max():.2f}')

# 3e. Nulos
nulos = dataset_v2.isnull().sum()
print(f'\n[5] Nulos: {nulos.sum()} ({"OK" if nulos.sum() == 0 else "WARN"})')

# 3f. 5 primeras y ultimas filas
print(f'\n[6] Primeras 5 filas (2019):')
display(dataset_v2.head(5))
print(f'\nUltimas 5 filas (2025):')
display(dataset_v2.tail(5))

VERIFICACIONES DE INTEGRIDAD

[1] Train features escaladas (deben tener mean~0, std~1):
  precio_chacra_kg                mean=-0.000000  std=1.0074  OK
  num_emergencias                 mean=+0.000000  std=1.0074  OK
  total_afectados                 mean=+0.000000  std=1.0074  OK
  hectareas_cultivo_perdidas      mean=+0.000000  std=1.0074  OK
  ALLSKY_SFC_SW_DWN               mean=-0.000000  std=1.0074  OK
  ... (17 columnas mas)

[2] Distribucion del TARGET (escalado) por periodo en train:
  2019-20 (24 meses): mean=1.2889  std=0.5266
  2021-24 (44 meses): mean=-0.7030  std=0.0123

[3] Features clave por periodo (escalado):
  precio_chacra_kg                19-20: mean=-0.755  21-24: mean=+0.412
  T2M                             19-20: mean=+0.347  21-24: mean=-0.189
  PRECTOTCORR                     19-20: mean=-0.994  21-24: mean=+0.542
  num_emergencias                 19-20: mean=+0.592  21-24: mean=-0.323

[4] Target en escala real (toneladas, media provincial mensual):
  Scal

,fecha_evento,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,...,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos,nlp_index,nlp_index_lag1,tiene_nlp,es_shock
0,2019-01-01,1.734878,-1.434221,0.035246,0.457959,-0.149083,-0.834618,-0.949641,1.463495,1.415830,...,0.658698,1.275497,-1.549276,-1.294024,1.373655,0.063457,0.304853,0.275792,0,0
1,2019-02-01,1.926381,-1.463101,3.400953,5.780756,0.065490,-1.708552,-0.949542,1.964423,1.823994,...,1.176964,0.757231,-1.255000,-1.294024,1.373655,0.063457,0.304853,0.275792,0,0
2,2019-03-01,1.854523,-1.367015,3.747470,1.312223,-0.139061,-1.106814,-0.953064,1.439828,1.742735,...,1.366662,0.049267,-0.960724,-1.294024,1.373655,0.063457,0.304853,0.275792,0,0
3,2019-04-01,1.489079,-1.438664,1.489601,0.137064,-0.148380,-0.443912,-0.992903,0.847549,1.202330,...,1.176964,-0.658698,-0.666448,-0.386873,-0.020502,-1.374911,0.304853,0.275792,0,0
4,2019-05-01,1.375385,-1.307784,-0.086734,-0.147731,-0.136714,-0.625489,-1.020537,0.034989,0.622101,...,0.658698,-1.176964,-0.372172,-0.386873,-0.020502,-1.374911,0.304853,0.275792,0,0



Ultimas 5 filas (2025):


,fecha_evento,produccion_t,precio_chacra_kg,num_emergencias,total_afectados,hectareas_cultivo_perdidas,ALLSKY_SFC_SW_DWN,PRECTOTCORR,QV2M,RH2M,...,month_sin,month_cos,mes_num,trimestre_num,trimestre_sin,trimestre_cos,nlp_index,nlp_index_lag1,tiene_nlp,es_shock
75,2025-04-01,-0.717114,0.341387,-0.913639,-0.537671,-0.131404,-0.602323,0.936490,0.743430,1.113147,...,1.176964,-0.658698,-0.666448,-0.386873,-0.020502,-1.374911,0.217506,1.061501,1,0
76,2025-05-01,-0.720850,0.249427,-0.913639,-0.537671,-0.131404,-1.229965,0.359915,0.178256,0.691013,...,0.658698,-1.176964,-0.372172,-0.386873,-0.020502,-1.374911,0.679231,0.183074,1,0
77,2025-06-01,-0.725318,0.244604,-0.913639,-0.537671,-0.131404,-2.285804,2.173861,-0.173110,0.913778,...,-0.049267,-1.366662,-0.077897,-0.386873,-0.020502,-1.374911,0.988349,0.673188,1,0
78,2025-07-01,-0.723668,0.409071,-0.913639,-0.537671,-0.131404,-0.317366,-0.545164,-1.269390,-0.303801,...,-0.757231,-1.176964,0.216379,0.520278,-1.414659,0.063457,1.100537,1.001313,1,0
79,2025-08-01,-0.725398,0.943308,-0.913639,-0.537671,-0.131404,0.333496,-0.378655,-1.152241,-0.823088,...,-1.275497,-0.658698,0.510655,0.520278,-1.414659,0.063457,0.660261,1.120398,1,0


## PASO 4 — Guardar dataset y scaler corregidos

In [6]:
# PASO 4: Guardar
output_path = ROOT / 'data/processed/master_escalado_extendido_v2.csv'
dataset_v2.to_csv(output_path, index=False)

# Guardar scaler v2
scaler_v2_params = {
    'features': COLS_SCALE,
    'mean': {col: float(m) for col, m in zip(COLS_SCALE, scaler_v2.mean_)},
    'scale': {col: float(s) for col, s in zip(COLS_SCALE, scaler_v2.scale_)},
    'target': TARGET,
    'target_mean': float(scaler_y_v2.mean_[0]),
    'target_scale': float(scaler_y_v2.scale_[0]),
    'n_train': n_train,
    'n_test': n_test,
    'train_range': f"{df_train['fecha_evento'].min().date()} -> {df_train['fecha_evento'].max().date()}",
    'test_range': f"{df_test['fecha_evento'].min().date()} -> {df_test['fecha_evento'].max().date()}",
    'correccion': 'Desnormalizacion de 2021-2025 con scaler fase2 original antes de re-normalizar',
}

scaler_path = ROOT / 'resultados/scaler_extendido_v2.json'
with open(scaler_path, 'w') as f:
    json.dump(scaler_v2_params, f, indent=2)

print(f'Dataset guardado: {output_path}')
print(f'  Tamano: {output_path.stat().st_size / 1024:.1f} KB')
print(f'Scaler guardado:  {scaler_path}')

# Estadisticas descriptivas del target por periodo
print(f'\n{"="*70}')
print(f'RESUMEN FINAL — TARGET POR PERIODO')
print(f'{"="*70}')

for label, mask_fn in [('2019-2020', lambda d: d.dt.year.isin([2019,2020])),
                        ('2021-2022', lambda d: d.dt.year.isin([2021,2022])),
                        ('2023-2024', lambda d: d.dt.year.isin([2023,2024])),
                        ('2025 (test)', lambda d: d.dt.year == 2025)]:
    mask = mask_fn(dataset_v2['fecha_evento'])
    if mask.sum() == 0:
        continue
    vals = dataset_v2.loc[mask, TARGET]
    vals_real = vals * scaler_y_v2.scale_[0] + scaler_y_v2.mean_[0]
    print(f'  {label:15s}  n={mask.sum():>3}  z-score: mean={vals.mean():+.3f} std={vals.std():.3f}  |  real: mean={vals_real.mean():.1f}t std={vals_real.std():.1f}t')

print(f'\n  Scaler target: mean={scaler_y_v2.mean_[0]:.2f}t  scale={scaler_y_v2.scale_[0]:.2f}t')
print(f'  Dataset: {len(dataset_v2)} filas x {len(dataset_v2.columns)} columnas')
print(f'{"="*70}')

Dataset guardado: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\data\processed\master_escalado_extendido_v2.csv
  Tamano: 36.9 KB
Scaler guardado:  C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-\resultados\scaler_extendido_v2.json

RESUMEN FINAL — TARGET POR PERIODO
  2019-2020        n= 24  z-score: mean=+1.289 std=0.527  |  real: mean=332.2t std=83.4t
  2021-2022        n= 24  z-score: mean=-0.703 std=0.010  |  real: mean=16.7t std=1.6t
  2023-2024        n= 24  z-score: mean=-0.703 std=0.014  |  real: mean=16.7t std=2.2t
  2025 (test)      n=  8  z-score: mean=-0.719 std=0.005  |  real: mean=14.1t std=0.8t

  Scaler target: mean=128.06t  scale=158.35t
  Dataset: 80 filas x 26 columnas
